## Table maintenance — `OPTIMIZE`, `ZORDER`, `VACUUM`

Three commands every Delta table needs over its life.

**`OPTIMIZE`** compacts many small files into fewer files near a ~1 GB target. Streaming writes and `MERGE` create lots of small files, and small files mean slow reads; `OPTIMIZE` is a metadata-level rewrite that cuts file count without changing what the table contains.

**`ZORDER BY (cols)`** co-locates rows with similar values into the same files, so file-skipping at read time gets dramatically better for queries that filter on those columns. `ZORDER BY (customer_id)` on `card_transactions` lets a per-customer lookup read one percent of the files instead of all of them.

**`VACUUM`** actually deletes the data files marked `remove` more than `deletedFileRetentionDuration` ago (default **7 days**). This is what reclaims storage. Delta **blocks** shorter retentions unless you explicitly override, because a short retention breaks in-flight readers and time travel.

```sql
OPTIMIZE silver.card_transactions ZORDER BY (customer_id, transaction_at);
VACUUM   silver.card_transactions RETAIN 168 HOURS;  -- 7 days = the default
```

On Unity Catalog managed tables, **Predictive Optimization** can run these for you automatically (module 08).